<a href="https://colab.research.google.com/github/Loopinlogix/Market_Analysis_Project-2/blob/main/Stock_Market_Project_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Stock Market Analysis Project 2


## Intro to this Project

This notebook is all about digging into some historical stock market data. We're gonna do a bunch of things: grab the data, clean it up (get rid of weird errors and bad values), find any crazy outliers, check for duplicates, cook up some new features from the existing data, make sure everything's on the same scale, and then split it all up so we can eventually build some machine learning models.

Basically, we've got two main data files: one with general info about stocks (`historical_stocks.csv`) like where they're traded, their names, what industry they're in, etc., and another with the daily prices and trading volumes (`historical_stock_prices.csv`).

The whole point here is to take all that raw, messy stock info and turn it into something neat and organized, packed with useful features. This way, we'll have a solid dataset ready to go for training models to try and figure out what the stock market might do next.

In [ ]:

#Github

#Github
!apt-get install -y git
!git config --global user.email "crystal_macneil@hotmail.com"
!git config --global user.name "Crystal MacNeil"

!git clone https://github.com/Loopinlogix/Market_Analysis_Project-2.git
%cd Market_Analysis_Project-2
!ls


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
git is already the newest version (1:2.34.1-1ubuntu1.17).
0 upgraded, 0 newly installed, 0 to remove and 3 not upgraded.
Cloning into 'Market_Analysis_Project-2'...
remote: Enumerating objects: 3, done.
remote: Counting objects: 100% (3/3), done.
remote: Total 3 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (3/3), done.
/content/Market_Analysis_Project-2
README.md


In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

print("=" * 60)
print("STEP 1: LOAD AND MERGE THE DATA")
print("=" * 60)

# Load datasets from Project 1
stocks = pd.read_csv("historical_stocks.csv")
prices = pd.read_csv("historical_stock_prices.csv", on_bad_lines='skip')

# Preview the raw data
print(prices.head())
print(stocks.head())
print(prices.info())
print(stocks.info())

# Clean column names (remove spaces, lowercase everything)
stocks.columns = stocks.columns.str.strip().str.lower()
prices.columns = prices.columns.str.strip().str.lower()

# === KEEP A TRULY RAW COPY FOR "BEFORE" COMPARISON PLOTS ===
prices_raw = prices.copy()

# Convert relevant columns to numeric, coercing errors
numeric_cols_to_convert = ['open', 'close', 'adj_close', 'low', 'high', 'volume']
for col in numeric_cols_to_convert:
    prices[col] = pd.to_numeric(prices[col], errors='coerce')

print(f"Stocks shape: {stocks.shape}")
print(f"Prices shape: {prices.shape}")

# Remove repeated header rows inside the dataset
initial_rows = len(stocks)
stocks = stocks[stocks["ticker"].str.upper() != "SYMBOL"]
stocks = stocks[stocks["ticker"].str.upper() != "TICKER"]

# Replace "N/A" with proper missing values
stocks["sector"] = stocks["sector"].replace("N/A", np.nan)
stocks["industry"] = stocks["industry"].replace("N/A", np.nan)

print(f"Removed {initial_rows - len(stocks)} repeated header rows")

# Merge stock info with price data
df = pd.merge(prices, stocks, on="ticker", how="left")
print(f"Combined dataset shape: {df.shape}")

print("=" * 60)
print("ADVANCED STEP 2: MISSING VALUE IMPUTATION")
print("=" * 60)

prices_adv = prices.copy()

# Convert date column to datetime
prices_adv["date"] = pd.to_datetime(prices_adv["date"], errors="coerce")

# Drop rows where date conversion failed
prices_adv = prices_adv.dropna(subset=["date"])

# Use date as index for time-based interpolation
prices_adv = prices_adv.set_index("date").sort_index()

# Function to interpolate missing numeric values
def advanced_impute(df):
    numeric_cols = ["open", "high", "low", "close", "volume"]

    # Time-based interpolation
    df[numeric_cols] = df[numeric_cols].interpolate(method="time")

    # Forward + backward fill
    df[numeric_cols] = df[numeric_cols].ffill().bfill()

    return df

# Apply imputation per ticker
prices_adv = prices_adv.groupby("ticker", group_keys=False).apply(advanced_impute)

print("Remaining missing values:\n", prices_adv.isnull().sum())

print("=" * 60)
print("ADVANCED STEP 3: OUTLIER DETECTION & CAPPING")
print("=" * 60)

# IQR-based capping function
def cap_iqr(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return series.clip(lower, upper)

# Apply capping to close price and volume
for col in ["close", "volume"]:
    prices_adv[col] = prices_adv.groupby("ticker")[col].transform(cap_iqr)

print("Outliers capped for close & volume.")

print("=" * 60)
print("ADVANCED STEP 4: ERROR CHECKING & CORRECTION")
print("=" * 60)

def fix_errors(df):
    numeric_cols = ["open", "high", "low", "close", "volume"]

    # Identify impossible or invalid values
    mask = (
        (df["open"] <= 0) |
        (df["high"] <= 0) |
        (df["low"] <= 0) |
        (df["close"] <= 0) |
        (df["volume"] < 0) |
        (df["high"] < df["low"])
    )

    # Replace errors with NaN
    df.loc[mask, numeric_cols] = np.nan

    # Re-interpolate to fix errors
    df[numeric_cols] = df[numeric_cols].interpolate(method="time").ffill().bfill()

    return df

prices_adv = prices_adv.groupby("ticker", group_keys=False).apply(fix_errors)
print("Errors corrected.")

print("=" * 60)
print("ADVANCED STEP 5: FEATURE ENGINEERING")
print("=" * 60)

prices_fe = prices_adv.copy()

# Daily returns
prices_fe["return"] = prices_fe.groupby("ticker")["close"].pct_change()

# 20-day moving average
prices_fe["ma_20"] = prices_fe.groupby("ticker")["close"].transform(lambda s: s.rolling(20).mean())

# 20-day volatility
prices_fe["vol_20"] = prices_fe.groupby("ticker")["return"].transform(lambda s: s.rolling(20).std())

# RSI indicator
def compute_rsi(series, period=14):
    delta = series.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain = gain.rolling(period).mean()
    avg_loss = loss.rolling(period).mean()
    rs = avg_gain / (avg_loss + 1e-9)
    return 100 - (100 / (1 + rs))

prices_fe["rsi_14"] = prices_fe.groupby("ticker")["close"].transform(compute_rsi)

print("Features added: return, ma_20, vol_20, rsi_14")

print("=" * 60)
print("ADVANCED STEP 6: NORMALIZATION")
print("=" * 60)


numeric_cols = [
    "open", "high", "low", "close", "volume",
    "return", "ma_20", "vol_20", "rsi_14"
]

# Drop rows with missing engineered features
prices_model = prices_fe.dropna(subset=numeric_cols).copy()

# Scale numeric fields
scaler = StandardScaler()
prices_model[numeric_cols] = scaler.fit_transform(prices_model[numeric_cols])

print("Numeric fields standardized.")

print("=" * 60)
print("ADVANCED STEP 7: ENCODING CATEGORICAL VARIABLES")
print("=" * 60)

cat_cols = [col for col in ['sector','industry'] if col in stocks.columns]

stocks_enc = pd.get_dummies(stocks, columns=cat_cols, drop_first=True)

prices_model_reset = prices_model.reset_index()

merged_final = pd.merge(prices_model_reset, stocks_enc, on='ticker', how='left')

print("Merged dataset shape:", merged_final.shape)


print("=" * 60)
print("ADVANCED STEP 8: DATA SPLITTING")
print("=" * 60)

# Sort by date to preserve time order
merged_final = merged_final.sort_values("date")

# Create target variable: next day's return
merged_final["target_next_return"] = merged_final.groupby("ticker")["return"].shift(-1)

# Remove rows without a target
merged_final = merged_final.dropna(subset=["target_next_return"])

# Select features
feature_cols = numeric_cols + [
    col for col in merged_final.columns if col.startswith("sector_") or col.startswith("industry_")
]

X = merged_final[feature_cols]
y = merged_final["target_next_return"]

# Split into train → validation → test
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, shuffle=False)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, shuffle=False)

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

print("="*75)
print("STEP 9: SAVE CLEAN DATA")
print("="*75)

merged_final.to_csv("stocks_clean_full.csv", index=False)
X_train.to_csv("X_train.csv", index=False)
X_val.to_csv("X_val.csv", index=False)
X_test.to_csv("X_test.csv", index=False)
y_train.to_csv("y_train.csv", index=False)
y_val.to_csv("y_val.csv", index=False)
y_test.to_csv("y_test.csv", index=False)

print("All cleaned datasets saved.")



STEP 1: LOAD AND MERGE THE DATA
  ticker   open  close  adj_close    low   high     volume        date
0    AHH  11.50  11.58   8.493155  11.25  11.68  4633900.0  2013-05-08
1    AHH  11.66  11.55   8.471151  11.50  11.66   275800.0  2013-05-09
2    AHH  11.55  11.60   8.507822  11.50  11.60   277100.0  2013-05-10
3    AHH  11.63  11.65   8.544494  11.55  11.65   147400.0  2013-05-13
4    AHH  11.60  11.53   8.456484  11.50  11.60   184100.0  2013-05-14
  ticker exchange                                    name             sector  \
0    PIH   NASDAQ  1347 PROPERTY INSURANCE HOLDINGS, INC.            FINANCE   
1  PIHPP   NASDAQ  1347 PROPERTY INSURANCE HOLDINGS, INC.            FINANCE   
2   TURN   NASDAQ                180 DEGREE CAPITAL CORP.            FINANCE   
3   FLWS   NASDAQ                 1-800 FLOWERS.COM, INC.  CONSUMER SERVICES   
4   FCCY   NASDAQ           1ST CONSTITUTION BANCORP (NJ)            FINANCE   

                     industry  
0  PROPERTY-CASUALTY INSURERS